# Week 2 — RAG-Powered Q&A System

**Knowledge base:** Think Python 2 (PDF) + Real Python Reference (Web) + PEP 8/20/257 (Text)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'Missing OPENAI_API_KEY in .env'

## Step 1: Load Documents

In [ ]:
from rag import load_all_documents

docs = load_all_documents()

by_type = {}
for d in docs:
    t = d.metadata.get('source_type', 'unknown')
    by_type[t] = by_type.get(t, 0) + 1

print(f'Total documents loaded: {len(docs)}')
for t, count in by_type.items():
    print(f'  {t}: {count} pages/docs')
print()
print('--- Sample: first PDF page ---')
print(docs[0].page_content[:500])

## Step 2: Chunk — Compare chunk_size 500 vs 1000

In [ ]:
from rag import chunk_documents

def chunk_stats(chunks, label):
    sizes = [len(c.page_content) for c in chunks]
    print(f'{label}: {len(chunks)} chunks | smallest={min(sizes)} | largest={max(sizes)}')

chunks_500  = chunk_documents(docs, chunk_size=500,  chunk_overlap=100)
chunks_1000 = chunk_documents(docs, chunk_size=1000, chunk_overlap=200)

chunk_stats(chunks_500,  'chunk_size=500 ')
chunk_stats(chunks_1000, 'chunk_size=1000')
print()
print('Observation: Smaller chunks = more granular retrieval but may split context mid-sentence.')
print('Larger chunks = more context per result but may include irrelevant content.')
print()
print('--- Sample chunk (chunk_size=1000) ---')
print(chunks_1000[10].page_content)

## Step 3: Embed + Store in ChromaDB

In [ ]:
from rag import build_vectorstore

# Using chunk_size=1000 for the main vector store
vectorstore = build_vectorstore(chunks_1000, persist=True)
print(f'Stored {len(chunks_1000)} chunks in ChromaDB')
print('Persisted to: ./chroma_db/')

## Step 4: Test Retrieval (before wiring up the LLM)

In [ ]:
test_queries = [
    'What is a list comprehension in Python?',
    'What does PEP 8 say about naming conventions?',
    'How do you define a function in Python?',
]

for query in test_queries:
    print(f'Query: {query}')
    results = vectorstore.similarity_search(query, k=3)
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get('source_name', '?')
        stype  = doc.metadata.get('source_type', '?')
        print(f'  [{i}] {source} ({stype}): {doc.page_content[:200]}...')
    print()

In [ ]:
print('Retrieval Relevance Annotation:')
print()
print('Q1 "list comprehension"')
print('  Relevant? YES — Think Python covers list comprehensions directly.')
print()
print('Q2 "PEP 8 naming conventions"')
print('  Relevant? YES — PEP 8 text is in the index and naming is a major section.')
print()
print('Q3 "define a function"')
print('  Relevant? YES — Think Python has dedicated chapters on functions.')

## Step 5: Build the RAG Chain

In [ ]:
from rag import create_chain

chain = create_chain(vectorstore)

for query in test_queries:
    result = chain.invoke({'query': query})
    print(f'Q: {query}')
    print(f'A: {result["result"]}')
    print()

## Step 6: Evaluate — 5-Question Eval Set

In [ ]:
eval_set = [
    {
        'question': 'What does the Zen of Python say about readability?',
        'expected_keywords': ['readability', 'counts'],
        'source_hint': 'pep',
    },
    {
        'question': 'How should docstrings be written according to PEP 257?',
        'expected_keywords': ['triple', 'quotes', 'docstring'],
        'source_hint': 'pep',
    },
    {
        'question': 'What is the difference between a parameter and an argument?',
        'expected_keywords': ['parameter', 'argument', 'function'],
        'source_hint': 'pdf',
    },
    {
        'question': 'What does PEP 8 recommend for maximum line length?',
        'expected_keywords': ['79', 'characters', 'line'],
        'source_hint': 'pep',
    },
    {
        'question': 'What is a tuple and how is it different from a list?',
        'expected_keywords': ['tuple', 'immutable', 'list'],
        'source_hint': 'pdf',
    },
]

results_table = []

for item in eval_set:
    result = chain.invoke({'query': item['question']})
    answer = result['result'].lower()
    chunks = result['source_documents']

    retrieved_types = [d.metadata.get('source_type', '') for d in chunks]
    retrieval_ok = item['source_hint'] in retrieved_types

    faithfulness_ok = any(
        kw in ' '.join(d.page_content.lower() for d in chunks)
        for kw in item['expected_keywords']
    )

    correctness_ok = any(kw in answer for kw in item['expected_keywords'])

    results_table.append({
        'question': item['question'],
        'answer': result['result'],
        'retrieval': retrieval_ok,
        'faithfulness': faithfulness_ok,
        'correctness': correctness_ok,
    })

print(f'{"Question":<55} {"Retrieval":<12} {"Faithful":<12} {"Correct"}')
print('-' * 95)
for r in results_table:
    ret = '✅' if r['retrieval']    else '❌'
    fth = '✅' if r['faithfulness'] else '❌'
    cor = '✅' if r['correctness']  else '❌'
    print(f'{r["question"][:54]:<55} {ret:<12} {fth:<12} {cor}')

print()
retrieval_score    = sum(r['retrieval']    for r in results_table)
faithfulness_score = sum(r['faithfulness'] for r in results_table)
correctness_score  = sum(r['correctness']  for r in results_table)
print(f'Retrieval:    {retrieval_score}/5')
print(f'Faithfulness: {faithfulness_score}/5')
print(f'Correctness:  {correctness_score}/5')

In [ ]:
print('Generated Answers:')
print()
for r in results_table:
    print(f'Q: {r["question"]}')
    print(f'A: {r["answer"]}')
    print()

## Stretch Goal A — Chunk Size Comparison

Running eval set with 3 different chunk sizes.

In [ ]:
from rag import chunk_documents, build_vectorstore, create_chain

chunk_configs = [300, 500, 1000]
comparison = {}

for size in chunk_configs:
    chunks = chunk_documents(docs, chunk_size=size, chunk_overlap=size // 5)
    vs = build_vectorstore(chunks, persist=False)
    ch = create_chain(vs)
    scores = {'retrieval': 0, 'faithfulness': 0, 'correctness': 0}
    for item in eval_set:
        res = ch.invoke({'query': item['question']})
        answer = res['result'].lower()
        retrieved_types = [d.metadata.get('source_type', '') for d in res['source_documents']]
        context_text = ' '.join(d.page_content.lower() for d in res['source_documents'])
        if item['source_hint'] in retrieved_types:
            scores['retrieval'] += 1
        if any(kw in context_text for kw in item['expected_keywords']):
            scores['faithfulness'] += 1
        if any(kw in answer for kw in item['expected_keywords']):
            scores['correctness'] += 1
    comparison[size] = scores
    print(f'chunk_size={size}: retrieval={scores["retrieval"]}/5 faithfulness={scores["faithfulness"]}/5 correctness={scores["correctness"]}/5')

best = max(comparison, key=lambda s: sum(comparison[s].values()))
print(f'\nBest performing chunk size: {best}')

## Stretch Goal E — Multi-Document (already implemented)

All three document types (PDF, Web, Text) are loaded in `load_all_documents()` and stored in the same ChromaDB vector store. Metadata fields `source_type` and `source_name` enable tracing which source each chunk came from.

## Observations

**What worked:**
- Multi-source loading was straightforward with LangChain loaders
- PEP text files gave precise retrieval for standards-based questions
- chunk_size=1000 with overlap=200 gave the best balance

**What didn't work / surprises:**
- Web loader for Real Python ref page returns mostly navigation/link text, not rich content — a scraper would do better
- chunk_size=300 split sentences mid-thought, hurting faithfulness
- PDF pages with figures/diagrams add noise (empty or garbled text)

**What I'd improve:**
- Use metadata filtering to prefer source type per query type
- Add BM25 hybrid search for keyword-heavy PEP queries
- Replace WebBaseLoader with a proper scraper for Real Python